# Model Controller Tutorial: EnviBert model (Multi Label)

> This notebook contains some example of how to use the EnviBert-based models in this NLP library

- skip_showdoc: true
- skip_exec: true

We will walk through other cases of classification: multi-head and multi-label. Since we will showcase the capabiilty of this label in these cases, there won't be as detailed as [this tutorial](https://anhquan0412.github.io/that-nlp-library/model_main_envibert.html)

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
#| hide
from nbdev.showdoc import *

## Load data

In [ ]:
from that_nlp_library.text_transformation import *
from that_nlp_library.text_augmentation import *
from that_nlp_library.text_main import *

In [ ]:
from underthesea import text_normalize
from functools import partial
from pathlib import Path
from importlib.machinery import SourceFileLoader
import os
from transformers import DataCollatorWithPadding


Define some necessary text augmentations and text transformations

> For Text Transformation

In [ ]:
txt_tfms=[text_normalize]

> For Text Augmentation

In [ ]:
over_nonown_tfm = partial(sampling_with_condition,query='Source=="non owned"',frac=0.5,seed=42,apply_to_all=False)
over_nonown_tfm.__name__ = 'Oversampling Non Owned'

over_own_tfm = partial(sampling_with_condition,query='Source=="owned"',frac=2,seed=42,apply_to_all=False)
over_own_tfm.__name__ = 'Oversampling Owned'

over_hc_tfm = partial(sampling_with_condition,query='Source=="hc search"',frac=2.5,seed=42,apply_to_all=False)
over_hc_tfm.__name__ = 'Oversampling HC search'

remove_accent_tfm = partial(remove_vnmese_accent,frac=1,seed=42,apply_to_all=True)
remove_accent_tfm.__name__ = 'Add No-Accent Text'

aug_tfms = [over_nonown_tfm,over_own_tfm,over_hc_tfm,remove_accent_tfm]

Create a TextDataMain object

In [ ]:
DATA_PATH = Path('secret_data')

In [ ]:
df = TextDataMain.from_csv(DATA_PATH/'buyer_listening_with_all_raw_data_w151617.csv',
                            return_df=True)

----- Input Validation Precheck -----
DataFrame contains missing values!
-----> List of columns and the number of missing values for each
is_valid    65804
dtype: int64
DataFrame contains duplicated values!
-----> Number of duplications: 7 rows


In [ ]:
df['L1L2'] = df[['L1','L2']].values.tolist()

In [ ]:
df.sample(5)

,Week,Group,Source,Content,L1,L2,L3,L4,is_valid,L1L2
28505,22.0,Google Play,Google Play,Tệ,Others,Cannot defined,-,-,NaN,"[Others, Cannot defined]"
65025,47.0,HC search,HC search,áo hiphop,Others,Cannot defined,-,-,NaN,"[Others, Cannot defined]"
3890,3.0,Google Play,Google Play,Lôi voucher,Feature,Apply Voucher,Voucher code issues,Tech,NaN,"[Feature, Apply Voucher]"
91483,35,1001 Hỏi đáp & tâm sự cùng shopee ✅,Non Owned,"Lên SHOPEE MALL không lo khóa sản phẩm, không ...",Others,Seller,,,0.0,"[Others, Seller]"
2813,2.0,Google Play,Google Play,"PHÍ SHIP QUÁ MẮC, KHÔNG THỂ NÀO CHẤP NHẬN ĐƯỢC...",Delivery,Shipping fee,Shipping fee,Non-tech,NaN,"[Delivery, Shipping fee]"


In [ ]:
tdm = TextDataMain(df,
                  main_content='Content',
                  metadatas='Source',
                  label_names='L1L2',
                  val_ratio=0.24,
                  split_cols='L1',
                  content_tfms = txt_tfms,
                  aug_tfms = aug_tfms,
                  process_metadatas=True,
                  seed=42,
                  shuffle_trn=True)

----- Input Validation Precheck -----
DataFrame contains missing values!
-----> List of columns and the number of missing values for each
is_valid    65804
dtype: int64
DataFrame contains duplicated values!
-----> Number of duplications: 7 rows


Define our tokenizer for EnviBert

In [ ]:
cache_dir=Path('./envibert_tokenizer')
tokenizer = SourceFileLoader("envibert.tokenizer", 
                             str(cache_dir/'envibert_tokenizer.py')).load_module().RobertaTokenizer(cache_dir)

EnviBert a data collator to work. We will save this as an attribute in TDM

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer,padding=True,max_length=512)
tdm.set_data_collator(data_collator)

Create our DatasetDict from TextDataMain (as our `ModelController` class can also work with DatasetDict)

In [ ]:
main_ddict= tdm.to_datasetdict(tokenizer,
                               max_length=512,
                               trn_ratio=0.1)

-------------------- Start Main Text Processing --------------------
----- Metadata Simple Processing & Concatenating to Main Content -----
----- Label Encoding -----
-------------------- Text Transformation --------------------
----- text_normalize -----


100%|████████████████████████████████████████████████████████████████████████████████| 112453/112453 [00:28<00:00, 3950.09it/s]


-------------------- Train Test Split --------------------
Previous Validation Percentage: 24.0%
- Before leak check
Size: 26989
- After leak check
Size: 23930
- Number of rows leaked: 3059, or 11.33% of the original validation (or test) data
Current Validation Percentage: 21.28%
-------------------- Text Augmentation --------------------
Train data size before augmentation: 88523
----- Oversampling Non Owned -----
Train data size after THIS augmentation: 98345
----- Oversampling Owned -----
Train data size after THIS augmentation: 109231
----- Oversampling HC search -----
Train data size after THIS augmentation: 116233
----- Add No-Accent Text -----


100%|███████████████████████████████████████████████████████████████████████████████| 116233/116233 [00:06<00:00, 19075.00it/s]


Train data size after THIS augmentation: 232466
Train data size after ALL augmentation: 232466
-------------------- Map Tokenize Function --------------------


Map:   0%|          | 0/23246 [00:00<?, ? examples/s]

Map:   0%|          | 0/23930 [00:00<?, ? examples/s]

In [ ]:
main_ddict

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'Source', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 23246
    })
    validation: Dataset({
        features: ['text', 'label', 'Source', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 23930
    })
})

In [ ]:
print(main_ddict['validation']['label'][:2])

[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]


# Model Experiment: EnviBert Multi-Head Classification (with Hidden Layer Concatenation)

In [ ]:
from that_nlp_library.models.classifiers import *
from that_nlp_library.model_main import *

comet_ml is installed but `COMET_API_KEY` is not set.


In [ ]:
from sklearn.metrics import f1_score, accuracy_score

In [ ]:
import os

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
# def accuracy_multi(inp, targ, thresh=0.5, sigmoid=True):
#     "Compute accuracy when `inp` and `targ` are the same size."
#     inp,targ = flatten_check(inp,targ)
#     if sigmoid: inp = inp.sigmoid()
#     return ((inp>thresh)==targ.bool()).float().mean()
# acc_metric = partial(accuracy_multi, thresh=0.5)

In [ ]:
# def skm_to_fastai(func, is_class=True, thresh=None, axis=-1, activation=None, **kwargs):
#     "Convert `func` from sklearn.metrics to a fastai metric"
#     dim_argmax = axis if is_class and thresh is None else None
#     if activation is None:
#         activation = ActivationType.Sigmoid if (is_class and thresh is not None) else ActivationType.No
#     return AccumMetric(func, dim_argmax=dim_argmax, activation=activation, thresh=thresh,
#                        to_np=True, invert_arg=True, **kwargs)

In [ ]:
# def F1ScoreMulti(thresh=0.5, sigmoid=True, labels=None, pos_label=1, average='macro', sample_weight=None):
#     "F1 score for multi-label classification problems"
#     activation = ActivationType.Sigmoid if sigmoid else ActivationType.No
#     return skm_to_fastai(skm.f1_score, thresh=thresh, activation=activation, flatten=False,
#                          labels=labels, pos_label=pos_label, average=average, sample_weight=sample_weight)

# f1_macro = F1ScoreMulti(thresh=0.5, average='macro')

## Train EnviBert (with hidden layer concatenation), using TDM

Let's create our model controller

In [ ]:
model_name='nguyenvulebinh/envibert'
num_classes = len(tdm.label_lists[0])

_model_kwargs={
    'concathead_class': RobertaConcatHeadSimple,
    'classifier_dropout':0.1,
    'last_hidden_size':768,  
    'is_multilabel':tdm.is_multilabel, 
    'is_multihead':tdm.is_multihead,
    'head_class_sizes': num_classes,
}

model = model_init_classification(model_class = RobertaHiddenStateConcatForSequenceClassification,
                                  cpoint_path = 'nguyenvulebinh/envibert', 
                                  output_hidden_states=True, # since we are using 'hidden layer contatenation'
                                  seed=42,
                                  model_kwargs = _model_kwargs)
metric_funcs = [partial(f1_score,average='macro'),accuracy_score]
controller = ModelController(model,tdm,metric_funcs)

Some weights of the model checkpoint at nguyenvulebinh/envibert were not used when initializing RobertaHiddenStateConcatForSequenceClassification: ['lm_head.bias', 'lm_head.dense.bias', 'lm_head.layer_norm.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing RobertaHiddenStateConcatForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaHiddenStateConcatForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaHiddenStateConcatForSequenceClassification were not initialized from the model checkpoint at nguyenvulebinh/envibert and are newly initialized: ['classification_head.out_proj.wei

And we can start training our model

In [ ]:
lr = 8.2e-5
bs=8
wd=0.01
epochs= 2

In [ ]:
controller.fit(epochs,lr,
               batch_size=bs,
               weight_decay=wd,
               save_checkpoint=False,
#                o_dir='sample_weights',
               compute_metrics=partial(compute_metrics_classification,
                                       is_multilabel=tdm.is_multilabel,
                                       multilabel_threshold=0.5)
              )

/home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/transformers/optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
/home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2372: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Epoch,Training Loss,Validation Loss,F1 Score L1l2,Accuracy Score L1l2
1,No log,0.046016,0.176854,0.481112
2,0.063600,0.040288,0.230808,0.554450


/home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2372: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [ ]:
controller.trainer.model.save_pretrained('./sample_weights/my_model')

## Predict using trained model, using TDM

### Load trained model

In [ ]:
model_name='nguyenvulebinh/envibert'
num_classes = len(tdm.label_lists[0])

_model_kwargs={
    'concathead_class': RobertaConcatHeadSimple,
    'classifier_dropout':0.1,
    'last_hidden_size':768,  
    'is_multilabel':tdm.is_multilabel, 
    'is_multihead':tdm.is_multihead,
    'head_class_sizes': num_classes,
}

model = model_init_classification(model_class = RobertaHiddenStateConcatForSequenceClassification,
                                  cpoint_path = 'sample_weights/my_model', 
                                  output_hidden_states=True, # since we are using 'hidden layer contatenation'
                                  seed=42,
                                  model_kwargs = _model_kwargs)
metric_funcs = [partial(f1_score,average='macro'),accuracy_score]
controller = ModelController(model,tdm,metric_funcs)

### Predict Train/Validation set

Make prediction on all validation set

In [ ]:
df_val = controller.predict_ddict(ds_type='validation',multilabel_threshold=0.5)

-------------------- Start making predictions --------------------


Map:   0%|          | 0/23930 [00:00<?, ? examples/s]

/home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2372: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [ ]:
df_val.head()

,text,label,Source,pred_L1L2,pred_prob_L1L2
0,owned - [ Cảnh báo ] bán fa.ke giả mạo Shop Ma...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...",owned,,"[0.0015464114, 0.001298852, 0.002373725, 0.001..."
1,google play - Chính sách trả hàng hoàn tiền kh...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",google play,"Dispute,Return/Refund","[0.0037290321, 0.0022984182, 0.0040654438, 0.0..."
2,google play - Hi vọng shopee kiểm duyệt phản h...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",google play,,"[0.001903328, 0.013303842, 0.039940067, 0.0028..."
3,google play - Shoppe bị lỗi r ....,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...",google play,"App performance,Feature","[0.000205651, 0.00082564016, 0.0013619855, 0.0..."
4,google play - Hàng không đặt được gì hết một sao,"[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ...",google play,"Cannot defined,Others","[0.00059689744, 0.004154379, 0.00455248, 0.000..."


In [ ]:
df_val.head()

,text,label,Source,pred_L1L2,pred_prob_L1L2
0,owned - [ Cảnh báo ] bán fa.ke giả mạo Shop Ma...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...",owned,,"[0.0015464114, 0.001298852, 0.002373725, 0.001..."
1,google play - Chính sách trả hàng hoàn tiền kh...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",google play,"Dispute,Return/Refund","[0.0037290321, 0.0022984182, 0.0040654438, 0.0..."
2,google play - Hi vọng shopee kiểm duyệt phản h...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",google play,,"[0.001903328, 0.013303842, 0.039940067, 0.0028..."
3,google play - Shoppe bị lỗi r ....,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...",google play,"App performance,Feature","[0.000205651, 0.00082564016, 0.0013619855, 0.0..."
4,google play - Hàng không đặt được gì hết một sao,"[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ...",google play,"Cannot defined,Others","[0.00059689744, 0.004154379, 0.00455248, 0.000..."


In [ ]:
df_val.pred_L1L2.value_counts()

pred_L1L2
Cannot defined,Others                     4894
App performance,Feature                   3863
                                          1994
Commercial,Promotions                     1292
Apply Voucher,Feature                     1193
                                          ... 
Cannot defined,Commercial,Promotions         1
Buyer complained seller,Cannot defined       1
Delivery,Seller                              1
Items/price                                  1
Delivery,Promotions,Shipping fee             1
Name: count, Length: 114, dtype: int64

To convert the label index to string, we can use the ```label_lists``` attribute of tdm

In [ ]:
import pandas as pd

In [ ]:
def get_label_str_multilabel(row):
    indices=np.where(row==1)[0]
    return ','.join([tdm.label_lists[0][i] for i in indices])
        

In [ ]:
df_val['label_str']=df_val['label'].apply(get_label_str_multilabel)

In [ ]:
df_val[['label_str','pred_L1L2']]

,label_str,pred_L1L2
0,"Buyer complained seller,Sellers packed fake or...",
1,"Dispute,Return/Refund","Dispute,Return/Refund"
2,"Feature,Review & Rating",
3,"Cannot defined,Others","App performance,Feature"
4,"Banned/blocked,Shopee account","Cannot defined,Others"
...,...,...
23925,"Order cancelled,Order/Item","Order cancelled,Order/Item"
23926,"Cannot defined,Others","Cannot defined,Others"
23927,"Shopee account,Sign up/Log in","App performance,Feature,Shopee account"
23928,"Cannot defined,Others","Delivery,Delivery time"


### Predict Test set

We will go through details on how to make a prediction on a completely new and raw dataset using our trained model. For now, let's reuse the sample csv and pretend it's our test set

In [ ]:
df_test = TextDataMain.from_csv(Path('sample_data')/'sample.csv',return_df=True)

----- Input Validation Precheck -----


We will remove all the labels and unnecessary columns

In [ ]:
df_test = df_test.drop(['L1','L2'],axis=1)

In [ ]:
df_test.head()

,Group,Source,Content
0,Google Play,Google Play,Mình khuyên các bạn nên mua bên Lazada hoặc Ti...
1,Google Play,Google Play,Con cc quoảng cáu ít thôi
2,iOS,iOS,Mình có một vài món hàng shipper ấn giao r mà ...
3,Google Play,Google Play,Mình đã sử dụng shoppe cũng 1 thời gian dài rồ...
4,Google Play,Google Play,Chăm sóc khách hàng quá tệ. Nhân viên hỗ trợ c...


We will create a DatasetDict for this test dataframe

In [ ]:
test_ddict = tdm.get_test_datasetdict_from_df(df_test)

-------------------- Getting Test Set --------------------
----- Input Validation Precheck -----
-------------------- Start Test Set Transformation --------------------
----- Metadata Simple Processing & Concatenating to Main Content -----
-------------------- Text Transformation --------------------
----- text_normalize -----


100%|████████████████████████████████████████████████████████████████████████████████████████| 70/70 [00:00<00:00, 5681.03it/s]

-------------------- Test Leak Checking --------------------
- Before leak check
Size: 70


- After leak check
Size: 0
- Number of rows leaked: 70, or 100.00% of the original validation (or test) data
-------------------- Construct DatasetDict --------------------


Map:   0%|          | 0/70 [00:00<?, ? examples/s]

Remember the ***Leak Check*** we did in TextDataMain? Our ```df_test``` only has 70 rows, and it also shows that 70 rows of our data is leaked (100%), which is correct because this test dataset is actually a small sample of the training data.

In [ ]:
test_ddict

DatasetDict({
    test: Dataset({
        features: ['text', 'Source', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 70
    })
})

Our test data has been processed + transformed (but not augmented) the same way as the validation set. Now we can start making the prediction

In [ ]:
controller = ModelController(model,tdm)
df_result = controller.predict_ddict(ddict=test_ddict,ds_type='test',multilabel_threshold=0.5)

-------------------- Start making predictions --------------------


Map:   0%|          | 0/70 [00:00<?, ? examples/s]

/home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2372: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [ ]:
df_result.head()

,text,Source,pred_L1L2,pred_prob_L1L2
0,google play - Mình khuyên các bạn nên mua bên ...,google play,,"[0.0018657269, 0.0076512336, 0.020795282, 0.00..."
1,google play - Con cc quoảng cáu ít thôi,google play,"Cannot defined,Others","[0.00033394128, 0.0001634884, 0.00032021094, 0..."
2,ios - Mình có một vài món hàng shipper ấn giao...,ios,"Delivery,Driver","[0.0028668162, 0.0037776325, 0.0012548639, 0.0..."
3,google play - Mình đã sử dụng shoppe cũng 1 th...,google play,,"[0.0020017219, 0.025779467, 0.017732793, 0.002..."
4,google play - Chăm sóc khách hàng quá tệ . Nhâ...,google play,"Contact Agent,Services","[0.0031769383, 0.015715336, 0.033517774, 0.002..."


If we just want to make a prediction on a small amount of data (single sentence, or a few sentences), we can use `ModelController.predict_raw_text`

In [ ]:
# Since we have some metadatas, we need to define a dictionary (to imitate a DatasetDict)
raw_content={
    'Source': 'Google play',
    'Content':'Tôi không thích Shopee.Tại vì dùng app rất chậm,lag banh nhà lầu, thậm chí log in còn không đc'
}

If we don't use metadata, we can use something like this: 

```raw_content='Tôi không thích Shopee.Tại vì dùng app rất chậm,lag banh nhà lầu, thậm chí log in còn không đc'```

In [ ]:
df_result = controller.predict_raw_text(raw_content,multilabel_threshold=0.5)
df_result

100%|██████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 4777.11it/s]


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

/home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2372: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


,text,Source,pred_L1L2,pred_prob_L1L2
0,google play - Tôi không thích Shopee . Tại vì ...,google play,"App performance,Feature","[0.0005422555, 0.0007667581, 0.0013118078, 0.0..."
